In [1]:
import pyomo.environ as pyo
import pandas as pd

In [2]:
# Creamos el modelo primal
m = pyo.ConcreteModel(name="Primal")

# Variables
m.x1 = pyo.Var(domain=pyo.NonNegativeReals)
m.x2 = pyo.Var(domain=pyo.NonNegativeReals)

# Restricciones
m.capacity = pyo.Constraint(expr=0.5*m.x1 + m.x2 >= 5)
m.flexibility = pyo.Constraint(expr=m.x1 + 0.5*m.x2 >= 4)

# Objetivo (Minimizar)
m.obj = pyo.Objective(expr=m.x1 + 1.5*m.x2, sense=pyo.minimize)

# Resolver
solver = pyo.SolverFactory('appsi_highs')
results = solver.solve(m)

print("--- SOLUCIÓN PRIMAL ---")
print(f"Estado: {results.solver.termination_condition}")
print(f"x1 = {pyo.value(m.x1):.4f}")
print(f"x2 = {pyo.value(m.x2):.4f}")
print(f"Z* (Costo Mínimo) = ${pyo.value(m.obj):.4f}")

--- SOLUCIÓN PRIMAL ---
Estado: optimal
x1 = 2.0000
x2 = 4.0000
Z* (Costo Mínimo) = $8.0000


In [5]:
# Creamos el modelo dual
d = pyo.ConcreteModel(name="Dual")

# Variables duales
d.y1 = pyo.Var(domain=pyo.NonNegativeReals)
d.y2 = pyo.Var(domain=pyo.NonNegativeReals)

# Restricciones
d.c1 = pyo.Constraint(expr=0.5*d.y1 + d.y2 <= 1)   # Asociada a x1
d.c2 = pyo.Constraint(expr=d.y1 + 0.5*d.y2 <= 1.5) # Asociada a x2

# Objetivo (Maximizar)
d.obj = pyo.Objective(expr=5*d.y1 + 4*d.y2, sense=pyo.maximize)

# Resolver
results_d = solver.solve(d)

print("\n--- SOLUCIÓN DUAL ---")
print(f"Estado: {results_d.solver.termination_condition}")
print(f"y1 = {pyo.value(d.y1):.4f}")
print(f"y2 = {pyo.value(d.y2):.4f}")
print(f"W* (Beneficio Máximo) = ${pyo.value(d.obj):.4f}")


--- SOLUCIÓN DUAL ---
Estado: optimal
y1 = 1.3333
y2 = 0.3333
W* (Beneficio Máximo) = $8.0000


Respuesta a tus preguntas:

¿Cuánto debería costar la capacidad efectiva? → $1.33 por unidad (es el recurso más escaso).

¿Cuánto debería costar la flexibilidad? → $0.33 por unidad (es el recurso menos escaso).

In [6]:
# --- PROBLEMA 2 ---
print("="*50)
print("PROBLEMA 2: Nuevos Coeficientes y RHS")
print("="*50)

m2 = pyo.ConcreteModel(name="Primal_2")
m2.x1 = pyo.Var(domain=pyo.NonNegativeReals)
m2.x2 = pyo.Var(domain=pyo.NonNegativeReals)

# Restricciones (RHS de capacidad cambió de 5 a 6)
m2.capacity = pyo.Constraint(expr=0.5*m2.x1 + m2.x2 >= 6)
m2.flexibility = pyo.Constraint(expr=m2.x1 + 0.5*m2.x2 >= 4)

# Nuevo Objetivo: x1 + 0.5*x2
m2.obj = pyo.Objective(expr=m2.x1 + 0.5*m2.x2, sense=pyo.minimize)

# Resolver
results_2 = solver.solve(m2)

print(f"x1 = {pyo.value(m2.x1):.4f}")
print(f"x2 = {pyo.value(m2.x2):.4f}")
print(f"Z* = ${pyo.value(m2.obj):.4f}")

PROBLEMA 2: Nuevos Coeficientes y RHS
x1 = 1.3333
x2 = 5.3333
Z* = $4.0000


In [7]:
d2 = pyo.ConcreteModel(name="Dual_2")
d2.y1 = pyo.Var(domain=pyo.NonNegativeReals)
d2.y2 = pyo.Var(domain=pyo.NonNegativeReals)

# Restricciones del dual (cambiaron los coeficientes de la función objetivo)
d2.c1 = pyo.Constraint(expr=0.5*d2.y1 + d2.y2 <= 1)   # Asociada a x1
d2.c2 = pyo.Constraint(expr=d2.y1 + 0.5*d2.y2 <= 0.5) # Asociada a x2

# Objetivo Dual (RHS del primal son los coeficientes)
d2.obj = pyo.Objective(expr=6*d2.y1 + 4*d2.y2, sense=pyo.maximize)

results_d2 = solver.solve(d2)

print(f"y1 (Capacidad) = {pyo.value(d2.y1):.4f}")
print(f"y2 (Flexibilidad) = {pyo.value(d2.y2):.4f}")
print(f"W* = ${pyo.value(d2.obj):.4f}")

y1 (Capacidad) = 0.0000
y2 (Flexibilidad) = 1.0000
W* = $4.0000


In [8]:
print("\n--- INTERPRETACIÓN PROBLEMA 2 ---")
print(f"Precio Sombra Capacidad Efectiva: ${pyo.value(d2.y1):.4f}")
print(f"Precio Sombra Flexibilidad:       ${pyo.value(d2.y2):.4f}")
print("\n¿Qué recurso es más escaso ahora?")
if pyo.value(d2.y1) > pyo.value(d2.y2):
    print("-> La Capacidad Efectiva es el recurso más escaso.")
else:
    print("-> La Flexibilidad es el recurso más escaso.")


--- INTERPRETACIÓN PROBLEMA 2 ---
Precio Sombra Capacidad Efectiva: $0.0000
Precio Sombra Flexibilidad:       $1.0000

¿Qué recurso es más escaso ahora?
-> La Flexibilidad es el recurso más escaso.


## Problema 3: Expansión de Generación y Transmisión (GTEP)

### Contexto
Este problema modela la planificación conjunta de inversiones en generación y transmisión en un sistema eléctrico de dos nodos, considerando los **precios nodales** (Locational Marginal Prices - LMP).

### Datos del Sistema

| Elemento | Nodo 1 | Nodo 2 |
|----------|--------|--------|
| **Costo de generación** | $c_1 = 17 \$/MWh$ | $c_2 = 25 \$/MWh$ |
| **Capacidad de generación** | $g_1 = 300$ MW | $g_2 = 300$ MW |
| **Demanda** | $d_1 = 200$ MW | $d_2 = 300$ MW |

*   **Línea de transmisión existente:** $F^0 = 50$ MW.
*   **Costos de inversión:**
    *   Nueva generación: $K_G^{inv} = 8 \$/MW$.
    *   Nueva transmisión: $K_L^{inv} = 5 \$/MW$.

### Variables de Decisión

*   $p_1, p_2$: Generación en cada nodo (MW).
*   $f$: Flujo de potencia por la línea (MW).
*   $\Delta g_1, \Delta g_2$: Expansión de capacidad de generación (MW).
*   $\Delta f$: Expansión de capacidad de transmisión (MW).

### Función Objetivo

Minimizar el costo total (operación + inversión):

$$
\min \sum_{i \in \{1,2\}} c_i p_i + K_G^{inv} (\Delta g_1 + \Delta g_2) + K_L^{inv} \Delta f
$$

### Restricciones

1.  **Balance Nodal:**
    $$
    p_1 - f = d_1
    $$
    $$
    p_2 + f = d_2
    $$

2.  **Límites de Generación:**
    $$
    0 \le p_1 \le g_1 + \Delta g_1
    $$
    $$
    0 \le p_2 \le g_2 + \Delta g_2
    $$

3.  **Límites de Transmisión:**
    $$
    -(F^0 + \Delta f) \le f \le (F^0 + \Delta f)
    $$

4.  **No Negatividad:**
    $$
    \Delta g_1, \Delta g_2, \Delta f \ge 0
    $$

### Resultados Esperados (Teoría)

*   **Costo Total:** \$10,350.00
*   **Generación:** Nodo 1 = 300 MW, Nodo 2 = 200 MW.
*   **Flujo (1→2):** 50 MW.
*   **Expansión de Transmisión:** 50 MW (la línea se satura).
*   **Precios Nodales:** Nodo 1 = \$20/MWh, Nodo 2 = \$25/MWh.

### Interpretación Económica

*   El **Nodo 1** es más barato ($c_1 = 17$), pero su capacidad está saturada (300 MW). Para exportar más, se necesita expandir la transmisión, lo que añade un costo marginal de \$5/MW. Por eso su precio nodal es $17 + 3 = 20 \$/MWh$ (el \$3 viene del dual de la restricción de capacidad de generación).
*   El **Nodo 2** es más caro ($c_2 = 25$), pero tiene holgura. Su precio nodal es simplemente su costo marginal: \$25/MWh.
*   La **expansión de transmisión** de 50 MW es rentable porque permite sustituir generación cara del Nodo 2 por generación barata del Nodo 1.

In [9]:
# --- PROBLEMA 3: GTEP con Precios Nodales ---
print("="*60)
print("PROBLEMA 3: Expansión de Generación y Transmisión")
print("="*60)

m3 = pyo.ConcreteModel(name="GTEP")

# --- Parámetros ---
c1, c2 = 17.0, 25.0
g1_max, g2_max = 300.0, 300.0
d1, d2 = 200.0, 300.0
F0 = 50.0
K_G_inv = 8.0
K_L_inv = 5.0

# --- Variables ---
# Generación
m3.p1 = pyo.Var(domain=pyo.NonNegativeReals)
m3.p2 = pyo.Var(domain=pyo.NonNegativeReals)
# Expansión de generación
m3.dg1 = pyo.Var(domain=pyo.NonNegativeReals)
m3.dg2 = pyo.Var(domain=pyo.NonNegativeReals)
# Flujo de transmisión
m3.f = pyo.Var(domain=pyo.Reals)  # Puede ser positivo o negativo
# Expansión de transmisión
m3.df = pyo.Var(domain=pyo.NonNegativeReals)

# --- Restricciones ---
# Balance Nodal
m3.balance1 = pyo.Constraint(expr=m3.p1 - m3.f == d1)
m3.balance2 = pyo.Constraint(expr=m3.p2 + m3.f == d2)

# Límites de generación
m3.gen1_limit = pyo.Constraint(expr=m3.p1 <= g1_max + m3.dg1)
m3.gen2_limit = pyo.Constraint(expr=m3.p2 <= g2_max + m3.dg2)

# Límites de transmisión
m3.flow_pos = pyo.Constraint(expr=m3.f <= F0 + m3.df)
m3.flow_neg = pyo.Constraint(expr=m3.f >= -(F0 + m3.df))

# --- Objetivo ---
m3.obj = pyo.Objective(
    expr=c1*m3.p1 + c2*m3.p2 + K_G_inv*(m3.dg1 + m3.dg2) + K_L_inv*m3.df,
    sense=pyo.minimize
)

# --- Resolver ---
results_3 = solver.solve(m3)

# --- Resultados ---
print(f"Estado: {results_3.solver.termination_condition}")
print(f"Costo Total: ${pyo.value(m3.obj):.2f}")
print(f"Generación Nodo 1: {pyo.value(m3.p1):.2f} MW")
print(f"Generación Nodo 2: {pyo.value(m3.p2):.2f} MW")
print(f"Flujo (1->2): {pyo.value(m3.f):.2f} MW")
print(f"Expansión Gen Nodo 1: {pyo.value(m3.dg1):.2f} MW")
print(f"Expansión Gen Nodo 2: {pyo.value(m3.dg2):.2f} MW")
print(f"Expansión Transmisión: {pyo.value(m3.df):.2f} MW")

PROBLEMA 3: Expansión de Generación y Transmisión
Estado: optimal
Costo Total: $10350.00
Generación Nodo 1: 300.00 MW
Generación Nodo 2: 200.00 MW
Flujo (1->2): 100.00 MW
Expansión Gen Nodo 1: 0.00 MW
Expansión Gen Nodo 2: 0.00 MW
Expansión Transmisión: 50.00 MW


In [12]:
# --- CELDA 10: PRECIOS NODALES (DUALES) ---
import pyomo.environ as pyo

# 1. Creamos el sufijo para importar los duales desde el solver
m3.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

# 2. Volvemos a resolver para que el solver llene el sufijo con los duales
solver = pyo.SolverFactory('appsi_highs')
results_3 = solver.solve(m3)

# 3. Extraemos los precios nodales (duales de las restricciones de balance)
print("\n--- PRECIOS NODALES (DUALES DE BALANCE) ---")
precio_nodo1 = pyo.value(m3.dual[m3.balance1])
precio_nodo2 = pyo.value(m3.dual[m3.balance2])

print(f"Precio Nodo 1: ${precio_nodo1:.2f}/MWh")
print(f"Precio Nodo 2: ${precio_nodo2:.2f}/MWh")

# 4. Mostramos también los duales de las restricciones de capacidad (opcional)
print("\n--- DUALES DE RESTRICCIONES DE CAPACIDAD ---")
print(f"Dual Gen 1 (límite): ${pyo.value(m3.dual[m3.gen1_limit]):.2f}")
print(f"Dual Gen 2 (límite): ${pyo.value(m3.dual[m3.gen2_limit]):.2f}")
print(f"Dual Flujo Positivo: ${pyo.value(m3.dual[m3.flow_pos]):.2f}")
print(f"Dual Flujo Negativo: ${pyo.value(m3.dual[m3.flow_neg]):.2f}")

# 5. Verificación de la función objetivo
print(f"\nCosto Total (Objetivo): ${pyo.value(m3.obj):.2f}")


--- PRECIOS NODALES (DUALES DE BALANCE) ---
Precio Nodo 1: $20.00/MWh
Precio Nodo 2: $25.00/MWh

--- DUALES DE RESTRICCIONES DE CAPACIDAD ---
Dual Gen 1 (límite): $-3.00
Dual Gen 2 (límite): $-0.00
Dual Flujo Positivo: $-5.00
Dual Flujo Negativo: $-0.00

Costo Total (Objetivo): $10350.00
